# IndicF5 Text-to-Speech (Colab GPU)

Runs AI4Bharat's [IndicF5](https://huggingface.co/ai4bharat/IndicF5) on a Colab GPU (CUDA).

**Before running:** In Colab, go to `Runtime > Change runtime type > GPU` (T4 is fine).

This notebook will:
1. Install dependencies
2. Log you into Hugging Face (the model is gated — you must accept the license once at https://huggingface.co/ai4bharat/IndicF5)
3. Load the model onto the GPU
4. Generate speech from text you provide
5. Print how long each stage took
6. Play and save the final audio

In [ ]:
!nvidia-smi

## 1. Install dependencies
Colab already ships a CUDA-enabled PyTorch. We pin `torch`/`torchaudio` to a matching `2.11.0` pair — mixing mismatched versions of the two (e.g. latest torch with an older torchaudio) can crash `torchaudio.transforms.MelSpectrogram` with an internal meta-device error.

In [ ]:
!pip install -q torch==2.11.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/ai4bharat/IndicF5.git soundfile huggingface_hub torchcodec
# Reinstall numpy last: the deps above (numba/librosa) can pull in a numpy build
# whose ABI doesn't match what's already loaded, causing:
#   ValueError: numpy.dtype size changed, may indicate binary incompatibility
!pip install -q --force-reinstall "numpy<2.3"

### ⚠️ Restart the runtime now

The cell below force-restarts the Python process so the freshly installed packages
(numpy in particular) are actually loaded — a `pip install` alone does **not**
reload already-imported binary extensions, which is what causes the
`numpy.dtype size changed, may indicate binary incompatibility` error.

After running it, Colab will say the session crashed/restarted — that's expected.
**Skip re-running the install cell above** and just continue from "2. Log into
Hugging Face" onward.

In [ ]:
import os
os.kill(os.getpid(), 9)

## 2. Log into Hugging Face
The model repo is gated. First visit https://huggingface.co/ai4bharat/IndicF5 and click **"Agree and access repository"** (one-time, requires a free HF account). Then get a token from https://huggingface.co/settings/tokens (read access is enough) and paste it below.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Load the model
We bypass `AutoModel.from_pretrained()`'s default construction path: `transformers>=5` always builds custom model classes inside a `torch.device("meta")` context, which breaks IndicF5's custom `INF5Model.__init__` (it eagerly builds real vocoder/DiT tensors). Instead we fetch the config/class via the dynamic-module machinery and instantiate the class directly, so `__init__` runs on the real device.

In [ ]:
import time
import torch
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

REPO_ID = "ai4bharat/IndicF5"

def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

def load_model_bypassing_meta_init():
    config = AutoConfig.from_pretrained(REPO_ID, trust_remote_code=True)
    model_cls = get_class_from_dynamic_module("model.INF5Model", REPO_ID)
    return model_cls(config)

device = get_device()
print(f"Using device: {device}")
if device != "cuda":
    print("WARNING: no CUDA GPU detected — check Runtime > Change runtime type > GPU in Colab.")

t0 = time.time()
model = load_model_bypassing_meta_init().to(device)
model_load_time = time.time() - t0
print(f"Model load time: {model_load_time:.1f}s")

## 4. Reference prompt audio
IndicF5 needs a short reference clip (speaker/prosody guide) + its transcript, in addition to the text you want spoken. We download one sample prompt from the upstream repo; swap in your own reference `.wav` + transcript if you like.

In [ ]:
import os
os.makedirs("prompts", exist_ok=True)

!wget -q -O prompts/PAN_F_HAPPY_00001.wav "https://raw.githubusercontent.com/AI4Bharat/IndicF5/main/prompts/PAN_F_HAPPY_00001.wav"

REF_AUDIO_PATH = "prompts/PAN_F_HAPPY_00001.wav"
REF_TEXT = "ਭਹੰਪੀ ਵਿੱਚ ਸਮਾਰਕਾਂ ਦੇ ਭਵਨ ਨਿਰਮਾਣ ਕਲਾ ਦੇ ਵੇਰਵੇ ਗੁੰਝਲਦਾਰ ਅਤੇ ਹੈਰਾਨ ਕਰਨ ਵਾਲੇ ਹਨ, ਜੋ ਮੈਨੂੰ ਖੁਸ਼ ਕਰਦੇ ਹਨ।"

## 5. Generate speech (timed)
Edit `TEXT` below to whatever you want spoken.

In [ ]:
TEXT = "नमस्ते! संगीत की तरह जीवन भी खूबसूरत होता है।"
OUT_PATH = "output.wav"

t0 = time.time()
audio = model(TEXT, ref_audio_path=REF_AUDIO_PATH, ref_text=REF_TEXT)
inference_time = time.time() - t0

import numpy as np
import soundfile as sf

if audio.dtype == np.int16:
    audio = audio.astype(np.float32) / 32768.0
sf.write(OUT_PATH, np.array(audio, dtype=np.float32), samplerate=24000)

audio_duration = len(audio) / 24000

print("=" * 50)
print(f"Model load time:   {model_load_time:.1f}s")
print(f"Inference time:    {inference_time:.1f}s")
print(f"Total time:        {model_load_time + inference_time:.1f}s")
print(f"Output audio dur:  {audio_duration:.1f}s")
print(f"Realtime factor:   {inference_time / audio_duration:.2f}x (compute-seconds per output-second)")
print("=" * 50)
print(f"Saved to: {OUT_PATH}")

## 7. Speed tuning: `nfe_step`

IndicF5 is a diffusion-style model — it takes `nfe_step` denoising steps per
generation (default **32**, hardcoded inside IndicF5's `forward()`). Fewer
steps means faster generation at some quality cost.

`INF5Model.forward()` doesn't expose `nfe_step` as a parameter, so this cell
reimplements the same logic as `forward()` (same pre/post-processing) but
calls `infer_process` directly so we can override it.

In [ ]:
import io
import numpy as np
import soundfile as sf
from pydub import AudioSegment, silence
from f5_tts.infer.utils_infer import infer_process, preprocess_ref_audio_text


def generate_with_nfe(model, text, ref_audio_path, ref_text, nfe_step=32,
                       cfg_strength=2.0, sway_sampling_coef=-1.0,
                       speed=1.0, remove_sil=True):
    """Same logic as INF5Model.forward(), but with nfe_step exposed."""
    ref_audio, ref_text = preprocess_ref_audio_text(ref_audio_path, ref_text)

    audio, final_sample_rate, _ = infer_process(
        ref_audio,
        ref_text,
        text,
        model.ema_model,
        model.vocoder,
        mel_spec_type="vocos",
        speed=speed,
        nfe_step=nfe_step,
        cfg_strength=cfg_strength,
        sway_sampling_coef=sway_sampling_coef,
        device=model.device,
    )

    buffer = io.BytesIO()
    sf.write(buffer, audio, samplerate=24000, format="WAV")
    buffer.seek(0)
    audio_segment = AudioSegment.from_file(buffer, format="wav")

    if remove_sil:
        non_silent_segs = silence.split_on_silence(
            audio_segment, min_silence_len=1000, silence_thresh=-50,
            keep_silence=500, seek_step=10,
        )
        audio_segment = sum(non_silent_segs, AudioSegment.silent(duration=0))

    target_dBFS = -20.0
    audio_segment = audio_segment.apply_gain(target_dBFS - audio_segment.dBFS)

    return np.array(audio_segment.get_array_of_samples())


# Benchmark a few step counts on the same text/reference
for steps in [32, 16, 8]:
    t0 = time.time()
    audio = generate_with_nfe(
        model, TEXT, REF_AUDIO_PATH, REF_TEXT, nfe_step=steps,
    )
    elapsed = time.time() - t0

    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    out_path = f"output_nfe{steps}.wav"
    sf.write(out_path, np.array(audio, dtype=np.float32), samplerate=24000)
    dur = len(audio) / 24000

    print(f"nfe_step={steps:>2}  |  time={elapsed:6.1f}s  |  audio_dur={dur:5.2f}s  |  realtime_factor={elapsed/dur:.2f}x  ->  {out_path}")

In [ ]:
from IPython.display import Audio, display

for steps in [32, 16, 8]:
    print(f"nfe_step={steps}")
    display(Audio(f"output_nfe{steps}.wav"))

## 6. Play the result

In [ ]:
from IPython.display import Audio, display
display(Audio(OUT_PATH))

## (Optional) Download the file to your computer

In [ ]:
from google.colab import files
files.download(OUT_PATH)